# Challenge 09: Re-Collecting All Data Is Inefficient

**PPT problem:** Re-collecting all data is inefficient  
**Technique:** Incremental loading with timestamp / cursor

A full refresh is simple but wasteful. Incremental loading stores a high-water mark such as last update timestamp or cursor, then asks the API only for new or changed data.


## Setup

Make sure the mock serving API is running:

```bash
cd day_2/day_2_1_API
python -m uvicorn mock_serving_api:app --reload --port 8011
```

If you use a different local port, update `BASE_URL` in the code cell.


## Run the Broken Program First

Do not fix the code before running it. Run it once and observe the failure signal.

Look for:

- Does the second run collect the same records again?
- Which timestamp can serve as a high-water mark?
- Where should the client store this value?



In [ ]:
"""
Challenge 09: Incremental loading.

This script runs the same full refresh twice. It works, but it recollects the
same weather records again. A robust client should store the last successful
update timestamp and send updated_since next time.
"""

import pandas as pd
import requests


BASE_URL = "http://127.0.0.1:8011"
CLIENT_ID = "group_09_incremental"


def fetch_weather_full_refresh() -> pd.DataFrame:
    page = 1
    all_records = []

    while True:
        response = requests.get(
            f"{BASE_URL}/debug-lab/09-incremental/weather/records",
            params={
                "page": page,
                "page_size": 20,
                "client_id": CLIENT_ID,
            },
            timeout=10,
        )
        response.raise_for_status()
        payload = response.json()
        all_records.extend(payload["data"])

        if payload["next_page"] is None:
            break
        page = payload["next_page"]

    return pd.DataFrame(all_records)


def main() -> None:
    first_run = fetch_weather_full_refresh()
    second_run = fetch_weather_full_refresh()
    combined = pd.concat([first_run, second_run], ignore_index=True)

    print("First run rows:", len(first_run))
    print("Second run rows:", len(second_run))
    print("Combined rows:", len(combined))
    print("Unique record_id values:", combined["record_id"].nunique())
    print("Expected: second run should collect only new or changed records.")
    print(combined[["record_id", "api_update_timestamp", "area", "forecast"]].head())


if __name__ == "__main__":
    main()


## Diagnose

Write short answers before changing the code.

1. Did the HTTP request fail, or did the workflow repeat work unnecessarily?
2. What row counts show that the second run recollected existing records?
3. Which timestamp can serve as the high-water mark for the next request?
4. Which technique from the PPT table should fix it?


In [ ]:
# Notes
# 1.
# 2.
# 3.
# 4.


## Where to Look for the Fix

Use the API docs, the observed failure, and the clues below to decide what to change.

Primary place to check: `http://127.0.0.1:8011/docs`, then find `/debug-lab/09-incremental/weather/records`.

Use these clues:

1. Open `/docs` and inspect whether the endpoint accepts an incremental parameter.
2. Store the latest successful `api_update_timestamp` after a successful run.
3. Send that timestamp as `updated_since` on the next run. If no newer records exist, an empty second result is a good outcome.

After that, use the success condition below to check whether your repaired code is good enough.


## Repair Workspace

Copy the broken code here and edit it until it works.

Success condition: The notebook should use `updated_since` so the second run avoids recollecting the same rows. If the source has not changed, the second run may correctly return zero rows.


In [ ]:
# Paste the broken code here, then repair it.
# Start by checking /docs for the endpoint contract and rereading the error output above.
# Stop when the success condition in the previous markdown cell is satisfied.
